# BCG GenAI Consulting — Financial Data Analysis
## GFC AI Chatbot Project: 10-K Data Extraction & Analysis

**Author:** Ayoub (Junior Data Scientist, BCG GenAI Consulting Team)  
**Manager:** Aisha (Senior Data Scientist)  
**Client:** Global Finance Corp. (GFC)  

---

### Objective
This notebook extracts and analyzes key financial data from the 10-K filings of **Microsoft**, **Tesla**, and **Apple** for the last three fiscal years (FY2021–FY2023). The goal is to identify significant financial trends and prepare the data for integration into an AI-powered financial chatbot.

### Data Sources
- **Microsoft**: FY ending June 30 — [SEC EDGAR](https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=MSFT&type=10-K)
- **Tesla**: FY ending December 31 — [SEC EDGAR](https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=TSLA&type=10-K)
- **Apple**: FY ending September 30 — [SEC EDGAR](https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=AAPL&type=10-K)

### Financial Metrics Extracted
- Total Revenue
- Net Income
- Total Assets
- Total Liabilities
- Cash Flow from Operating Activities

All figures are in **billions of USD**.

---
## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Libraries loaded successfully.')

In [ ]:
# -----------------------------------------------------------------------
# Manually extracted data from SEC EDGAR 10-K filings
# Sources:
#   Microsoft 10-K FY2021/2022/2023 — filed with SEC, CIK 0000789019
#   Tesla 10-K FY2021/2022/2023     — filed with SEC, CIK 0001318605
#   Apple 10-K FY2021/2022/2023     — filed with SEC, CIK 0000320193
# All values in billions USD
# -----------------------------------------------------------------------

data = {
    'Company': [
        # Microsoft
        'Microsoft', 'Microsoft', 'Microsoft',
        # Tesla
        'Tesla', 'Tesla', 'Tesla',
        # Apple
        'Apple', 'Apple', 'Apple'
    ],
    'Fiscal Year': [
        2021, 2022, 2023,
        2021, 2022, 2023,
        2021, 2022, 2023
    ],
    'Total Revenue (B)': [
        168.09, 198.27, 211.92,   # Microsoft
          53.82,  81.46,  97.69,   # Tesla
         365.82, 394.33, 383.29    # Apple
    ],
    'Net Income (B)': [
         61.27,  72.74,  72.36,   # Microsoft
          5.52,  12.56,  14.97,   # Tesla
         94.68,  99.80,  96.99    # Apple
    ],
    'Total Assets (B)': [
        333.78, 364.84, 411.98,   # Microsoft
         62.13,  82.34, 106.62,   # Tesla
        351.00, 352.76, 352.58    # Apple
    ],
    'Total Liabilities (B)': [
        191.79, 198.30, 205.75,   # Microsoft
         30.55,  36.44,  43.00,   # Tesla
        287.91, 302.08, 290.44    # Apple
    ],
    'Operating Cash Flow (B)': [
         76.74,  89.04,  87.91,   # Microsoft
         11.50,  14.48,  13.26,   # Tesla
        104.04, 122.15, 114.00    # Apple
    ]
}

df = pd.DataFrame(data)
df

---
## 2. Derived Metrics & Feature Engineering

In [ ]:
# Sort to ensure correct ordering for pct_change()
df = df.sort_values(['Company', 'Fiscal Year']).reset_index(drop=True)

# --- Year-over-year growth rates ---
df['Revenue Growth (%)']      = df.groupby('Company')['Total Revenue (B)'].pct_change() * 100
df['Net Income Growth (%)']   = df.groupby('Company')['Net Income (B)'].pct_change() * 100
df['Asset Growth (%)']        = df.groupby('Company')['Total Assets (B)'].pct_change() * 100
df['Op. CF Growth (%)']       = df.groupby('Company')['Operating Cash Flow (B)'].pct_change() * 100

# --- Financial health ratios ---
df['Net Profit Margin (%)']   = (df['Net Income (B)'] / df['Total Revenue (B)']) * 100
df['Debt Ratio']              = df['Total Liabilities (B)'] / df['Total Assets (B)']
df['Shareholders Equity (B)'] = df['Total Assets (B)'] - df['Total Liabilities (B)']
df['Op. CF Margin (%)']       = (df['Operating Cash Flow (B)'] / df['Total Revenue (B)']) * 100

df

---
## 3. Company-Level Summaries

In [ ]:
summary_cols = [
    'Company', 'Fiscal Year',
    'Total Revenue (B)', 'Net Income (B)',
    'Net Profit Margin (%)', 'Debt Ratio',
    'Operating Cash Flow (B)', 'Revenue Growth (%)'
]

for company in ['Microsoft', 'Tesla', 'Apple']:
    print(f'\n{'='*60}')
    print(f'  {company}')
    print(f'{'='*60}')
    print(df[df['Company'] == company][summary_cols].to_string(index=False))

---
## 4. Trend Analysis — Visualizations

### 4.1 Total Revenue

In [ ]:
fig, ax = plt.subplots()
for company in ['Microsoft', 'Tesla', 'Apple']:
    sub = df[df['Company'] == company]
    ax.plot(sub['Fiscal Year'], sub['Total Revenue (B)'], marker='o', label=company, linewidth=2)

ax.set_title('Total Revenue (FY2021–FY2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Revenue (Billions USD)')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}B'))
plt.tight_layout()
plt.show()

### 4.2 Net Income

In [ ]:
fig, ax = plt.subplots()
for company in ['Microsoft', 'Tesla', 'Apple']:
    sub = df[df['Company'] == company]
    ax.plot(sub['Fiscal Year'], sub['Net Income (B)'], marker='s', label=company, linewidth=2)

ax.set_title('Net Income (FY2021–FY2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Net Income (Billions USD)')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}B'))
plt.tight_layout()
plt.show()

### 4.3 Net Profit Margin

In [ ]:
fig, ax = plt.subplots()
for company in ['Microsoft', 'Tesla', 'Apple']:
    sub = df[df['Company'] == company]
    ax.plot(sub['Fiscal Year'], sub['Net Profit Margin (%)'], marker='^', label=company, linewidth=2)

ax.set_title('Net Profit Margin % (FY2021–FY2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Net Profit Margin (%)')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
plt.tight_layout()
plt.show()

### 4.4 Revenue Growth Year-over-Year

In [ ]:
growth_df = df[df['Fiscal Year'] > 2021][['Company', 'Fiscal Year', 'Revenue Growth (%)']]

fig, ax = plt.subplots()
companies = growth_df['Company'].unique()
x = [2022, 2023]
width = 0.25
offsets = [-width, 0, width]

for i, company in enumerate(companies):
    vals = growth_df[growth_df['Company'] == company]['Revenue Growth (%)'].values
    bars = ax.bar([xi + offsets[i] for xi in x], vals, width=width, label=company)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_title('Revenue Growth YoY % (FY2022 & FY2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Growth (%)')
ax.set_xticks(x)
ax.legend()
plt.tight_layout()
plt.show()

### 4.5 Debt Ratio

In [ ]:
fig, ax = plt.subplots()
for company in ['Microsoft', 'Tesla', 'Apple']:
    sub = df[df['Company'] == company]
    ax.plot(sub['Fiscal Year'], sub['Debt Ratio'], marker='D', label=company, linewidth=2)

ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.4, label='50% threshold')
ax.set_title('Debt Ratio (Total Liabilities / Total Assets) FY2021–FY2023',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Debt Ratio')
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

### 4.6 Operating Cash Flow

In [ ]:
fig, ax = plt.subplots()
for company in ['Microsoft', 'Tesla', 'Apple']:
    sub = df[df['Company'] == company]
    ax.bar(
        sub['Fiscal Year'] + ['Microsoft', 'Tesla', 'Apple'].index(company) * 0.25 - 0.25,
        sub['Operating Cash Flow (B)'],
        width=0.25, label=company
    )

ax.set_title('Operating Cash Flow (FY2021–FY2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Operating Cash Flow (Billions USD)')
ax.set_xticks([2021, 2022, 2023])
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}B'))
plt.tight_layout()
plt.show()

---
## 5. Cross-Company Comparison (FY2023)

In [ ]:
fy2023 = df[df['Fiscal Year'] == 2023][[
    'Company', 'Total Revenue (B)', 'Net Income (B)',
    'Net Profit Margin (%)', 'Debt Ratio',
    'Operating Cash Flow (B)', 'Op. CF Margin (%)',
    'Shareholders Equity (B)'
]].set_index('Company')

print('FY2023 Cross-Company Comparison')
print('=' * 70)
fy2023

---
## 6. Key Findings & Financial Health Summary

### Microsoft
**Revenue trend:** Consistent growth — $168B (FY2021) → $212B (FY2023), a **26% increase** over three years.  
**Profitability:** Net profit margin remained exceptionally high (~34%), indicating strong cost discipline and high-margin software/cloud revenues.  
**Balance sheet:** Assets grew steadily from $334B to $412B; debt ratio stable (~50%), suggesting controlled leverage.  
**Cash generation:** Operating cash flow reached $87.9B in FY2023 — among the highest globally.  
**Overall:** Microsoft demonstrates **stable, high-margin growth** driven by its Azure cloud and enterprise software segments. Financially very healthy.

---

### Tesla
**Revenue trend:** Explosive growth — $54B (FY2021) → $98B (FY2023), an **81% increase** in two years.  
**Profitability:** Net profit margin grew from ~10% to ~15%, reflecting improving operational efficiency despite heavy capital expenditure.  
**Balance sheet:** Total assets nearly doubled ($62B → $107B), while the debt ratio remained low (~40%), showing asset-light scaling.  
**Cash generation:** Operating cash flow grew but **dipped slightly in FY2023** ($14.5B → $13.3B), suggesting margin pressure from price cuts and increased competition.  
**Overall:** Tesla shows **hyper-growth with improving but pressured profitability**. High growth trajectory makes it attractive but volatile.

---

### Apple
**Revenue trend:** Peak in FY2022 ($394B), slight decline in FY2023 ($383B) — reflecting iPhone demand softening and macro headwinds.  
**Profitability:** Net profit margin ~25%, consistently high, driven by premium pricing and services growth.  
**Balance sheet:** Total assets remarkably stable (~$352B), but **debt ratio is high (~82%)** — Apple actively uses leverage for share buybacks, a deliberate capital strategy.  
**Cash generation:** Best-in-class operating cash flow ($114B in FY2023), enabling massive shareholder returns.  
**Overall:** Apple is a **cash-generating machine** with a mature but slightly slowing top line. High debt ratio is intentional and manageable given cash generation.

---

## 7. Implications for the AI Chatbot

The structured data prepared in this notebook forms the foundation for the GFC AI chatbot. Key design considerations:

1. **Trend queries:** The chatbot should handle questions like *"How has Tesla's revenue grown over the last 3 years?"* — addressed by the YoY growth columns.
2. **Comparative queries:** *"Which company has the highest profit margin?"* — the FY2023 cross-company table enables this.
3. **Health assessment:** *"Is Apple's debt level concerning?"* — the debt ratio and context (buyback strategy) allow nuanced answers.
4. **Cash flow analysis:** Operating cash flow margin is a critical indicator that differentiates operational quality from accounting profits.

The dataset is clean, consistently formatted (all values in billions USD), and ready for ingestion into an NLP-based financial chatbot pipeline.

In [ ]:
# Export clean dataset to CSV for chatbot ingestion
df.to_csv('financial_data_clean.csv', index=False)
print('Clean dataset exported to financial_data_clean.csv')
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print('\nColumns:', list(df.columns))